#### Login to Huggingface

In [ ]:
from huggingface_hub import login
login("YOUR_HUGGINGFACE_TOKEN")

In [ ]:
!pip install -U peft


#### Base Model

In [ ]:
# import torch
# from transformers import AutoTokenizer, AutoModelForCausalLM

# MODEL_ID = "google/gemma-2b-it"

# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
# model = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID,
#     torch_dtype=torch.float16,
#     device_map="auto"
# )

#### Fine-tuned Model

In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

base = AutoModelForCausalLM.from_pretrained(
    "google/gemma-2b-it",
    torch_dtype=torch.float16,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained("gemma-lora-webq-finetuned")

model = PeftModel.from_pretrained(base, "gemma-lora-webq-finetuned")
model.config.pad_token_id = tokenizer.pad_token_id


model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


2025-07-28 16:18:30.652872: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Loading checkpoint shards: 100%|██████████| 2/2 [00:13<00:00,  6.82s/it]
/opt/anaconda3/envs/nemo_env/lib/python3.9/site-packages/bitsandbytes/cextension.py:34: UserWarning: The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers, 8-bit multiplication, and GPU quantization are unavailable.
  warn("The installed version of bitsandbytes was compiled without GPU support. "


'NoneType' object has no attribute 'cadam32bit_grad_fp32'


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): GemmaForCausalLM(
      (model): GemmaModel(
        (embed_tokens): Embedding(256000, 2048, padding_idx=0)
        (layers): ModuleList(
          (0-17): 18 x GemmaDecoderLayer(
            (self_attn): GemmaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj):

#### Load Knowledge Graph

In [8]:
import pandas as pd
df = pd.read_csv("knowledge_graph.csv")

filtered_df = df[["subject_label", "predicate_label", "object_label"]].copy()

filtered_df['subject_label'] = filtered_df['subject_label'].astype(str).str.strip()
filtered_df['predicate_label'] = filtered_df['predicate_label'].astype(str).str.strip()
filtered_df['object_label'] = filtered_df['object_label'].astype(str).str.strip()

filtered_df['subject_label_lower'] = filtered_df['subject_label'].str.lower()
filtered_df['predicate_label_lower'] = filtered_df['predicate_label'].str.lower()

#### Grab Unique Subjects & Predicts From Knowledge Graph

In [9]:
subject_list = filtered_df['subject_label_lower'].unique()
predicate_list = filtered_df['predicate_label_lower'].unique()

In [ ]:
!pip install sentence_transformers

In [ ]:
!pip install sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer, util

# Load a lightweight embedding model
embedder = SentenceTransformer("all-mpnet-base-v2")

filtered_df["triple_text"] = (
    filtered_df["subject_label"] + " | "
    + filtered_df["predicate_label"] + " | "
    + filtered_df["object_label"]
)

triples = filtered_df["triple_text"].tolist()

# Precompute embeddings once
triple_embeddings = embedder.encode(
    triples, convert_to_tensor=True, show_progress_bar=True, normalize_embeddings= True
)

Batches: 100%|██████████| 3718/3718 [06:12<00:00,  9.97it/s]


#### Semantic search (Cosine Similarity)

In [1]:
def get_semantic_facts(question, triples, triple_embeddings, top_k=3):
    
    q_emb = embedder.encode(question, convert_to_tensor=True, normalize_embeddings=True)
    cos_scores = util.cos_sim(q_emb, triple_embeddings)[0]
    top_idxs = torch.topk(cos_scores, k=top_k).indices.tolist()

    print([triples[i] for i in top_idxs])

    return [triples[i] for i in top_idxs]

def get_facts_for_question(question):
    try:
        return get_semantic_facts(question, triples, triple_embeddings, top_k=5)
    except Exception:
        return []

#### Extract Facts from Knowledge Graph

In [ ]:
# import difflib

# def extract_best_match(text, candidates, cutoff=0.6):
#     text = text.lower()

#     best_matches = difflib.get_close_matches(text, candidates, n=1, cutoff=cutoff)
#     if best_matches:
#         return best_matches[0]
    
#     matches = [c for c in candidates if c in text]
#     return max(matches, key=len) if matches else None

# def get_facts_for_question(question, df, subject_list, predicate_list):

#     matched_subject = extract_best_match(question, subject_list)
#     if matched_subject is None:
#         return []

#     subject_df = df[df["subject_label_lower"] == matched_subject]
#     if subject_df.empty:
#         return []

#     matched_predicate = extract_best_match(question, predicate_list)

#     if matched_predicate:

#         filtered_df = subject_df[
#             subject_df["predicate_label"].str.lower() == matched_predicate
#         ]
#         if not filtered_df.empty:
#             facts = [
#                 f"{row['subject_label']} — {row['predicate_label']} — {row['object_label']}"
#                 for _, row in filtered_df.iterrows()
#             ]
#             return facts

#     facts = [
#         f"{row['subject_label']} — {row['predicate_label']} — {row['object_label']}"
#         for _, row in subject_df.iterrows()
#     ]
#     return facts


#### Generate Answer using facts from Knowledge Graph

In [2]:
def generate_answer(question, facts):
    formatted_facts = "\n".join(f"- {fact}" for fact in facts)
    prompt = f"""<bos>
[INST]
Using the facts below, answer the question with a short, direct answer.

Facts:
{formatted_facts}

Question: {question}
[/INST]
"""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False
    )
    
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if "[/INST]" in full_output:
        answer_only = full_output.split("[/INST]")[-1].strip()
    else:
        answer_only = full_output.strip()

    return answer_only


In [ ]:
# question = "what state does selena gomez?"

# facts = get_facts_for_question(question, filtered_df, subject_list, predicate_list)

# answer = generate_answer(question, facts)

# print(answer)

#### Questions from Webquestions Dataset (20%)

In [3]:
from datasets import load_dataset
import pandas as pd

dataset = load_dataset("stanfordnlp/web_questions", split="test")

full = load_dataset("stanfordnlp/web_questions")
full = full.filter(lambda ex: len(ex["answers"]) > 0)
split = full["train"].train_test_split(test_size=0.2, seed=42)
raw_train, raw_test = split["train"], split["test"]

questions_df = pd.DataFrame(raw_test)[['question', 'answers']]

/opt/anaconda3/envs/nemo_env/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Filter: 100%|██████████| 2032/2032 [00:00<00:00, 77256.89 examples/s]


#### Process One Question

In [4]:
def process_question(question):
    # facts = get_facts_for_question(question, filtered_df, subject_list, predicate_list)
    facts = get_facts_for_question(question)
    
    if not facts:
        facts = ["No facts available."]
    return generate_answer(question, facts)

#### Answer all questions

In [5]:
from tqdm import tqdm

tqdm.pandas()

questions_df['predicted_answer'] = questions_df['question'].progress_apply(process_question)

  0%|          | 1/756 [00:00<00:01, 506.07it/s]


NameError: name 'tokenizer' is not defined

In [ ]:
question = "Where was Justin Bieber born?"

print("Question:", question)

facts = get_facts_for_question(question)
print("\nTop-K Retrieved Facts:")
for fact in facts:
    print("-", fact)

answer = generate_answer(question, facts)
print("\nGenerated Answer:", answer)

# Optional: Show reference if available
ref_row = questions_df[questions_df['question'] == question]
if not ref_row.empty:
    print("\nReference Answer:", ref_row['answers'].values[0])
else:
    print("\nReference Answer not found.")

#### Prepare Data for Evaluation

In [ ]:
references_bleu = questions_df['answers'].tolist()
references_rouge = questions_df['answers'].apply(lambda x: ' / '.join(x)).tolist()

predictions = questions_df['predicted_answer'].tolist()

#### Evaluate Predictions

In [ ]:
import evaluate

bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")

bleu_res = bleu.compute(predictions=predictions, references=references_bleu)
rouge_res = rouge.compute(predictions=predictions, references=references_rouge)

results_df = pd.DataFrame([{
    "bleu":      bleu_res["bleu"],
    "rouge1":    rouge_res["rouge1"],
    "rouge2":    rouge_res["rouge2"],
    "rougeL":    rouge_res["rougeL"],
    "rougeLsum": rouge_res["rougeLsum"],
}])

results_df.to_csv("evaluation_results.csv", index=False)

#### Evaluate Predictions with WebQuestions Dataset

In [ ]:
import json

In [ ]:
def clean_prediction(text: str) -> str:
    """Remove any leading 'Answer:' (case-insensitive) and extra whitespace."""
    text = text.strip()
    if text.lower().startswith("answer:"):
        return text.split(":", 1)[1].strip()
    return text

In [ ]:
def answer_base_model(question):
    prompt = f"<bos>\n[INST]\nQuestion: {question}\n[/INST]"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    outputs = base.generate(**inputs, max_new_tokens=256, do_sample=False)
    raw = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # get only after the [/INST] if present
    ans = raw.split("[/INST]")[-1].strip()
    return clean_prediction(ans)

In [ ]:
def answer_finetuned_model(question):
    # re-use your generate_answer with empty facts
    ans = generate_answer(question, [])
    return clean_prediction(ans)

In [ ]:
def eval_and_dump(ds, answer_fn, filename):
    records = []
    for ex in ds:
        q = ex["question"]
        records.append({
            "question":   q,
            "answers":    ex["answers"],
            "prediction": answer_fn(q)
        })
    with open(filename, "w") as f:
        json.dump(records, f, indent=2)
    print(f"Wrote {filename} ({len(records)} records)")

In [ ]:
for split_name, ds in [("train", raw_train), ("test", raw_test)]:
    eval_and_dump(ds, answer_base,    f"base_{split_name}_predictions.json")
    eval_and_dump(ds, answer_ft,      f"finetuned_{split_name}_predictions.json")
    eval_and_dump(ds, answer_ftkg,    f"finetuned_kg_{split_name}_predictions.json")

In [ ]:
import json
import time
import openai
from openai.error import RateLimitError, OpenAIError

# assume client = openai.OpenAI(api_key=config.OPENAI_API_KEY) is already set

def evaluate_with_gpt4o(question, prediction, references, retries=3, backoff=1.0):
    system = {
        "role": "system",
        "content": (
            "You are a strict QA evaluator.  "
            "Given a question, a list of reference answers, and a model's single predicted answer, "
            "determine whether the predicted answer is correct.  "
            "Return EXACTLY a JSON object with two fields:\n"
            "\"correct\": true or false\n"
            "\"rationale\": a brief explanation of why it is correct or not"
        )
    }
    user = {
        "role": "user",
        "content": json.dumps({
            "question": question,
            "reference_answers": references,
            "model_prediction": prediction
        }, ensure_ascii=False)
    }

    for attempt in range(1, retries + 1):
        try:
            resp = client.chat.completions.create(
                model="gpt-4o",
                messages=[system, user],
                temperature=0.0,
            )
            text = resp.choices[0].message.content.strip()

            # if GPT didn’t return valid JSON, force a retry
            if not text.startswith("{"):
                raise ValueError("Non-JSON response")

            return json.loads(text)

        except RateLimitError:
            wait = backoff * (2 ** (attempt - 1))
            print(f"[RateLimit] retry {attempt}/{retries} after {wait:.1f}s")
            time.sleep(wait)

        except (ValueError, json.JSONDecodeError) as e:
            # malformed or missing JSON
            wait = backoff
            print(f"[ParseError] {e!r}: retry {attempt}/{retries} after {wait:.1f}s")
            time.sleep(wait)

        except OpenAIError as e:
            # other API errors—maybe transient
            wait = backoff * (2 ** (attempt - 1))
            print(f"[OpenAIError] {e!r}: retry {attempt}/{retries} after {wait:.1f}s")
            time.sleep(wait)

    # if all retries fail, return a safe default
    return {
        "correct": False,
        "rationale": "Could not evaluate (rate limits or parse errors)."
    }


In [ ]:
# 1) Install dependencies (once):
#    pip install transformers accelerate bitsandbytes

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import json
import time

# 2) Load your open-source chat model
model_name = "meta-llama/Llama-2-13b-chat-hf"
tokenizer  = AutoTokenizer.from_pretrained(model_name)
model      = AutoModelForCausalLM.from_pretrained(
    model_name,
    load_in_8bit=True,            # quantize to 8-bit to fit on a single GPU
    device_map="auto"
)

chat = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    do_sample=False
)

# 3) Redefine your evaluator to use this local model
def evaluate_with_local_llm(question, prediction, references, retries=2):
    prompt = f"""
You are a strict QA evaluator.  
Given a question, a list of reference answers, and a model’s single predicted answer,  
determine whether the predicted answer is correct.

Return EXACTLY a JSON object with two fields:
  "correct": true or false,
  "rationale": a brief explanation of why it is correct or not.

Question: {question}
Reference Answers: {references}
Model Prediction: {prediction}
"""
    for attempt in range(retries):
        out = chat(prompt)[0]["generated_text"]
        # extract JSON substring
        start = out.find("{")
        end   = out.rfind("}") + 1
        if start != -1 and end != -1:
            try:
                return json.loads(out[start:end])
            except json.JSONDecodeError:
                pass
        time.sleep(1)  # simple retry backoff

    # fallback
    return {"correct": False, "rationale": "Evaluation failed or timed out."}

# 4) Replace your loop over prediction_files with this
prediction_files = [
    "base_test_predictions.json",
    "finetuned_test_predictions.json",
    "finetuned_kg_test_predictions.json",
    # …and train files if you want
]

for fname in prediction_files:
    with open(fname) as f:
        records = json.load(f)

    out_records = []
    for rec in records:
        verdict = evaluate_with_local_llm(
            rec["question"],
            rec["prediction"],   # or rec["predicted_answer"] as you named it
            rec["answers"]
        )
        rec.update(verdict)
        out_records.append(rec)

    out_name = f"local_evaluated_{fname}"
    with open(out_name, "w", encoding="utf-8") as f:
        json.dump(out_records, f, indent=2, ensure_ascii=False)
    print(f"Wrote {out_name} ({len(out_records)} entries)")
